# v4r5 QLoRA training (Colab T4)

This notebook trains Qwen3-4B on the clean, broad v4r5 dataset. It intentionally stops after decode calibration. Do not run litmus or `blind_v4r5` until the decoding setting is locked.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select the Colab T4 GPU kernel first.'
print(torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
!pip install -q unsloth trl datasets textstat

## Train v4r5

Commit and push the finished dataset and code before running this cell. Colab clones GitHub, so uncommitted local files are invisible to the remote kernel.

In [ ]:
%cd /content
![ -d SmallLearningModel ] || git clone https://github.com/Elitelord/SmallLearningModel.git
%cd SmallLearningModel
!git pull --ff-only
!python -m data.audit_v4r3 data/v4/gold_v4_r5.jsonl --accuracy-gate clean-v2 --forbid-targeted-v4r3
!python -m train.qlora_train \
    --data data/v4/gold_v4_r5.jsonl \
    --adapter-name v4r5 \
    --epochs 2 \
    --lora-r 16 --lora-alpha 32 \
    --batch-size 8 --grad-accum 2 --lr 1e-4

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5')
backup_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(Path('train/adapters/v4r5'), backup_dir / 'adapter', dirs_exist_ok=True)
print(f'Adapter backed up to {backup_dir / "adapter"}')

## Decode calibration

This uses only `calibration_v4r5`, never the litmus or blind holdout. Choose temperature by average readability across every seed, not by the best individual seed.

In [ ]:
!python -m eval.tuned_sweep \
    --adapter train/adapters/v4r5 \
    --eval-key calibration_v4r5 \
    --temperatures 0,0.3,0.7 --seeds 0,1,2 \
    --top-settings 7 \
    --out eval/v4r5_decode_calibration.json

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
backup_dir = Path('/content/drive/MyDrive/SmallLearningModel/v4r5')
backup_dir.mkdir(parents=True, exist_ok=True)
for result_path in Path('eval').glob('v4r5_decode_calibration*.json'):
    shutil.copy2(result_path, backup_dir / result_path.name)
print(f'Calibration files backed up to {backup_dir}')

## Stop here

Share the calibration table before running held-out evaluation. The adapter and calibration JSON are now backed up under `MyDrive/SmallLearningModel/v4r5`.